# Chronologically Consistent LLMs: Stock Return Prediction
This notebook replicates the stock return prediction task using the FNSPID dataset (2019-2022). We will extract news embeddings using distinct checkpoints of ChronoBERT (2018 and 2022) to observe the impact of look-ahead bias or chronological consistency.

We use a universe of 100 highly liquid stocks. The strategy fits a rolling Ridge regression to predict next-day returns based on news embeddings, then forms a daily zero-cost, long-short portfolio (going long the top decile and short the bottom decile).

In [ ]:
import os

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import torch
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import Ridge
from tqdm import tqdm
import matplotlib.pyplot as plt

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
MODELS = {
    "ChronoBERT_2017": "manelalab/chrono-bert-v1-20171231",
    "ChronoBERT_2018": "manelalab/chrono-bert-v1-20181231",
    # "ChronoBERT_2019": "manelalab/chrono-bert-v1-20191231",
    # "ChronoBERT_2020": "manelalab/chrono-bert-v1-20201231",
    "ChronoBERT_2021": "manelalab/chrono-bert-v1-20211231",
    "ChronoBERT_2022": "manelalab/chrono-bert-v1-20221231",
    #"ChronoBERT_2023": "manelalab/chrono-bert-v1-20231231",
    "Standard_BERT": "bert-base-uncased",
    "FinBERT": "yiyanghkust/finbert-pretrain"
}

In [ ]:
# 1. Load the FNSPID dataset
# Columns used: Date, Stock_symbol, Article_title
df_news = pd.read_csv("dataset/fnspid_2019_2022.csv")
df_news['date'] = pd.to_datetime(df_news['Date']).dt.date

# 2. Define the top 100 significant stocks (Example: top S&P 100 tickers)
# For brevity, a subset is shown here. You can expand this list to exactly 100.
# top_100_tickers = [
#     'AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOGL', 'GOOG', 'META', 'BRK-B', 'LLY', 'AVGO',
#     'V', 'JPM', 'TSLA', 'WMT', 'UNH', 'MA', 'PG', 'JNJ', 'HD', 'ORCL',
#     'MRK', 'COST', 'ABBV', 'BAC', 'CRM', 'CVX', 'NFLX', 'AMD', 'KO', 'PEP',
#     'TMO', 'LIN', 'DIS', 'MCD', 'WFC', 'CSCO', 'ABT', 'INTU', 'IBM', 'GE',
#     'CAT', 'AXP', 'DHR', 'QCOM', 'TXN', 'VZ', 'AMGN', 'INTC', 'COP', 'NEE',
#     'PFE', 'BA', 'SPGI', 'HON', 'UNP', 'RTX', 'LMT', 'LOW', 'GS', 'SYK',
#     'BKNG', 'ELV', 'MS', 'PLD', 'MDT', 'C', 'BLK', 'ADP', 'TJX', 'BMY',
#     'MMC', 'CB', 'AMT', 'SCHW', 'GILD', 'SBUX', 'MDLZ', 'CI', 'ISRG', 'DE',
#     'MO', 'SO', 'DUK', 'CMCSA', 'T', 'REGN', 'ZTS', 'VRTX', 'NOC', 'EOG',
#     'BDX', 'SLB', 'BSX', 'EMR', 'CL', 'AON', 'CSX', 'USB', 'F', 'XOM'
# ]

# Covid-19 affected stocks
top_100_tickers = [
    'ZM', 'PTON', 'DOCU', 'NFLX', 'AMZN', 'SHOP', 'TDOC', 'NVDA', 'AMD', 'AAPL',
    'MSFT', 'GOOGL', 'META', 'ETSY', 'CHWY', 'W', 'ROKU', 'PYPL', 'SQ', 'TEAM',
    'CRWD', 'OKTA', 'ZS', 'TWLO', 'DDOG', 'FSLY', 'NET', 'SNOW', 'EBAY', 'CRM',
    'TGT', 'WMT', 'COST', 'HD', 'LOW', 'CLX', 'PG', 'KMB', 'DPZ', 'MRNA',
    'PFE', 'BNTX', 'JNJ', 'GILD', 'REGN', 'ABT', 'LLY', 'AZN', 'NVAX', 'CCL',
    'RCL', 'NCLH', 'DAL', 'UAL', 'AAL', 'LUV', 'BA', 'MAR', 'HLT', 'EXPE',
    'BKNG', 'ABNB', 'UBER', 'LYFT', 'DIS', 'AMC', 'CNK', 'LYV', 'MGM', 'CZR',
    'WYNN', 'LVS', 'DRI', 'PLAY', 'SYY', 'XOM', 'CVX', 'SLB', 'HAL', 'OXY',
    'SPG', 'VNO', 'BXP', 'MAC', 'GPS', 'M', 'JWN', 'KSS', 'URBN', 'RTX',
    'GE', 'JPM', 'BAC', 'WFC', 'C', 'AXP', 'V', 'MA', 'SBUX', 'TSLA'
]

# Filter news for our specific stock universe
df_news = df_news[df_news['Stock_symbol'].isin(top_100_tickers)].copy()
df_news.sort_values('date', inplace=True)
print(f"Total news articles for top tickers: {len(df_news)}")

In [ ]:
# Fetch daily adjusted close prices to calculate next-day forward returns (T+1)
start_date = "2019-01-01"
end_date = "2023-01-01"

prices = yf.download(top_100_tickers, start=start_date, end=end_date)['Close']

# Calculate daily returns
returns = prices.pct_change()

# Shift returns by -1 to get the T+1 forward return (what we want to predict)
# If news comes out on day T, we trade at the close of T and hold until close of T+1
forward_returns = returns.shift(-1)

# Reshape forward returns to long format for easy merging
fwd_ret_long = forward_returns.reset_index().melt(id_vars='Date', var_name='Stock_symbol', value_name='fwd_return')
fwd_ret_long['Date'] = fwd_ret_long['Date'].dt.date
fwd_ret_long.rename(columns={'Date': 'date'}, inplace=True)

# Merge returns with news data
df_merged = pd.merge(df_news, fwd_ret_long, on=['date', 'Stock_symbol'], how='inner')
df_merged.dropna(subset=['fwd_return'], inplace=True)

In [ ]:
df_merged.head()

### Feature Extraction via ChronoBERT
We load multiple models from Hugging Face. 
*Note: Replace the model string paths with the exact repository names provided by the paper's authors if they differ.*

In [ ]:
def extract_embeddings(texts, model_name, batch_size=32):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()
    
    embeddings = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=f"Embedding with {model_name}"):
            batch_texts = texts[i:i+batch_size].tolist()
            inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
            outputs = model(**inputs)
            # Use the [CLS] token embedding as the sentence representation
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(cls_embeddings)
            
    return np.vstack(embeddings)

# Extract embeddings for all models
for model_name, model_path in MODELS.items():
    df_merged[model_name] = list(extract_embeddings(df_merged['Article_title'].fillna(""), model_path))

### Strategy Backtesting: Fama-MacBeth Rolling Regression
We train a Ridge regression model dynamically. To avoid look-ahead bias in the regression itself, we use a rolling window of historical news embeddings to predict the out-of-sample next-day returns. 

For a universe of 100 stocks, a decile strategy means we go long the top 10 stocks with the highest predicted returns, and short the bottom 10.

In [ ]:
from collections import Counter

def backtest_strategy_with_long_short_tickers(df, embedding_col, lookback_days=60):
    unique_dates = sorted(df['date'].unique())
    portfolio_returns = []
    
    # Trackers for both legs of the portfolio
    long_tickers_history = [] 
    short_tickers_history = [] 
    
    for i in range(lookback_days, len(unique_dates)):
        train_dates = unique_dates[i-lookback_days:i]
        test_date = unique_dates[i]
        
        train_data = df[df['date'].isin(train_dates)]
        test_data = df[df['date'] == test_date]
        
        if len(train_data) < 100 or len(test_data) < 20: 
            continue
            
        X_train = np.vstack(train_data[embedding_col].values)
        y_train = train_data['fwd_return'].values
        X_test = np.vstack(test_data[embedding_col].values)
        
        # Fit Ridge regression
        model = Ridge(alpha=1.0)
        model.fit(X_train, y_train)
        
        # Predict next day's returns
        predictions = model.predict(X_test)
        test_data = test_data.copy()
        test_data['predicted_return'] = predictions
        
        # Aggregate predictions by ticker
        daily_preds = test_data.groupby('Stock_symbol').agg({
            'predicted_return': 'mean',
            'fwd_return': 'mean' 
        })
        
        n_stocks = len(daily_preds)
        k = max(1, int(n_stocks * 0.10)) 
        
        daily_preds = daily_preds.sort_values('predicted_return', ascending=False)
        long_portfolio = daily_preds.head(k)
        short_portfolio = daily_preds.tail(k)
        
        # Track both long and short tickers
        long_tickers_history.extend(long_portfolio.index.tolist())
        short_tickers_history.extend(short_portfolio.index.tolist())
        
        # Equal-weighted portfolio return
        daily_return = long_portfolio['fwd_return'].mean() - short_portfolio['fwd_return'].mean()
        portfolio_returns.append(daily_return)
        
    return pd.Series(portfolio_returns), long_tickers_history, short_tickers_history

# Run backtests
returns = []
for model_name in MODELS.keys():
    # 1. Run the updated backtests (this will return both the returns and the lists of tickers)
    returns_pd, longs, shorts = backtest_strategy_with_long_short_tickers(df_merged, model_name)
    returns.append(returns_pd)
    
    # 2. Count the occurrences of each ticker in the long/short history
    # We convert the list to a pandas Series and use value_counts() to rank them
    top_longs = pd.Series(longs).value_counts().head(10)
    top_shorts = pd.Series(shorts).value_counts().head(10)
    
    print(f"\nTop 10 Most Frequent Long Positions for {model_name}:")
    print(top_longs)

    # print(f"\nTop 10 Most Frequent Short Positions for {model_name}:")
    # print(top_shorts)

### Evaluation Metrics
We measure the annualized Sharpe ratio to evaluate the risk-adjusted returns. Because this is a self-financing, market-neutral strategy, the formula simplifies to:

$$\text{Sharpe Ratio} = \frac{\sqrt{252} \cdot \mu}{\sigma}$$

Where $\mu$ is the mean daily portfolio return and $\sigma$ is the standard deviation of daily returns.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

def calculate_sharpe(returns_series):
    mean_ret = returns_series.mean()
    std_ret = returns_series.std()
    return np.sqrt(252) * (mean_ret / std_ret) if std_ret > 0 else 0

print("--- Strategy Performance ---")

fig, ax = plt.subplots(figsize=(12, 7))

trading_dates = sorted(df_merged['date'].unique())[60:]
for i in range(len(returns)):
    returns[i].index = pd.to_datetime(trading_dates)
    sharpe = calculate_sharpe(returns[i])
    print(f"{list(MODELS)[i]} Annualized Sharpe Ratio: {sharpe:.3f}")

    # Calculate cumulative returns
    cum_returns = (1 + returns[i]).cumprod() - 1
    
    # Plot the cumulative returns
    ax.plot(cum_returns.index, cum_returns, label=f'{list(MODELS)[i]}')

# --- NEW: Add shaded region for the COVID-19 Market Crash ---
# We use pd.to_datetime to ensure the dates match the x-axis format
start_crash = pd.to_datetime('2020-02-19')
end_crash = pd.to_datetime('2020-03-23')

# axvspan creates the vertical shading. 
# 'alpha' controls the transparency (0 is invisible, 1 is solid)
ax.axvspan(start_crash, end_crash, color='red', alpha=0.15, label='COVID-19 Crash')

ax.set_title('Cumulative Returns of Long-Short Portfolio')
ax.set_xlabel('Date')
ax.set_ylabel('Cumulative Return')

# Format the x-axis to display Year-Month and auto-rotate the labels
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
fig.autofmt_xdate() 

# Place the legend in the upper left so it doesn't overlap the latest data points
ax.legend(loc='upper left')
ax.grid(True)
plt.show()